# PLN Sesión 2: Candidatos y Significados (Terminología y Vectores)

---

## ¿Por qué?

En la sesión anterior se analizó cómo el modelo identifica entidades individuales (quién es quién). Sin embargo, el lenguaje especializado rara vez se manifiesta en palabras sueltas. Conceptos como *inteligencia artificial*, *red social* o *aprendizaje automático* son unidades de significado compuestas por múltiples elementos.

Además, persistía un desafío: ¿cómo saber que dos términos distintos, como *bulo* y *fake news*, se refieren a lo mismo si no comparten ninguna característica gráfica? Esta sesión aborda la transición de la forma al concepto mediante la extracción de terminología compleja y el análisis de similitud semántica.

## ¿Qué?

Se trabajarán dos conceptos fundamentales:
1. **Extracción terminológica basada en patrones:** El uso de categorías gramaticales (POS) para identificar combinaciones frecuentes que constituyen términos técnicos.
2. **Vectores de palabras (Word Embeddings):** La representación matemática de las palabras en un espacio multidimensional donde la cercanía geométrica equivale a la cercanía de significado.

## ¿Para qué?

Para un profesional de la lengua, estas técnicas permiten:
- **Crear glosarios automáticos:** Identificar candidatos a términos técnicos en grandes volúmenes de texto especializado.
- **Agrupar por significado:** Encontrar sinónimos o conceptos relacionados sin necesidad de reglas de búsqueda exactas.
- **Analizar campos semánticos:** Entender qué conceptos orbitan alrededor de un término clave (ej. qué palabras se asocian con *tecnología* en un corpus determinado).

## ¿Cómo?

Se empleará **spaCy** para filtrar patrones morfosintácticos y para explorar la distancia vectorial entre términos, superando definitivamente la "dictadura de la forma" impuesta por los motores de búsqueda tradicionales.

---
## 1. Preparación del entorno

Cada sesión de trabajo requiere la inicialización del modelo de lengua. En este caso, se emplea la versión reducida (`sm`), optimizada para tareas de análisis morfosintáctico y extracción de patrones.

In [ ]:
!python -m spacy download es_core_news_sm --quiet
import spacy
import pandas as pd
from collections import Counter

procesador = spacy.load("es_core_news_sm")
print("Modelo de lengua española cargado.")

---
## 2. Extracción de terminología compleja

La mayoría de los términos técnicos no son monolemas (una palabra), sino polilemas (varias palabras). Las estructuras más comunes en español son **Sustantivo + Adjetivo** e **Híper-sustantivos** (Sustantivo + Preposición + Sustantivo).

In [ ]:
texto_tecnico = """
La inteligencia artificial y el aprendizaje automático están transformando la lingüística computacional.
El procesamiento de lenguaje natural requiere una infraestructura digital robusta y modelos de lengua 
entrenados con corpus lingüísticos de alta calidad. La brecha tecnológica se reduce con innovación abierta.
"""

documento = procesador(texto_tecnico)

candidatos = []

for posicion in range(len(documento)):
    token = documento[posicion]
    
    # Patrón A: Sustantivo + Adjetivo (ej. inteligencia artificial)
    if posicion + 1 < len(documento):
        siguiente = documento[posicion + 1]
        if token.pos_ == "NOUN" and siguiente.pos_ == "ADJ":
            candidatos.append(f"{token.text} {siguiente.text}")
            
    # Patrón B: Sustantivo + Preposición + Sustantivo (ej. modelos de lengua)
    if posicion + 2 < len(documento):
        preposicion = documento[posicion + 1]
        segundo_sustantivo = documento[posicion + 2]
        if token.pos_ == "NOUN" and preposicion.pos_ == "ADP" and segundo_sustantivo.pos_ == "NOUN":
            candidatos.append(f"{token.text} {preposicion.text} {segundo_sustantivo.text}")

print("Candidatos a términos encontrados:")
for candidato in candidatos:
    print(f"- {candidato}")

---
### 2.1 Bajo el capó: qué etiqueta asigna el modelo

Antes de poder detectar patrones, el modelo analiza cada token y le asigna una **categoría gramatical** (POS, del inglés *Part-of-Speech*). Las etiquetas son universales: el mismo sistema funciona para español, inglés o francés.

La celda siguiente muestra, para el texto del ejercicio anterior, qué categoría asigna el modelo a cada palabra.

| Código | Categoría | Ejemplo |
|--------|-----------|---------|
| `NOUN` | Sustantivo | *lengua*, *corpus*, *modelo* |
| `PROPN` | Nombre propio | *Madrid*, *Google*, *España* |
| `ADJ` | Adjetivo | *artificial*, *computacional*, *robusta* |
| `VERB` | Verbo | *analiza*, *transforma*, *requiere* |
| `AUX` | Verbo auxiliar | *ha*, *puede*, *está* |
| `ADV` | Adverbio | *radicalmente*, *muy*, *también* |
| `ADP` | Preposición | *de*, *en*, *con*, *para* |
| `DET` | Determinante | *el*, *una*, *los*, *esta* |
| `PRON` | Pronombre | *él*, *esto*, *que*, *se* |
| `CCONJ` | Conjunción coordinante | *y*, *o*, *pero* |
| `NUM` | Número | *tres*, *2024*, *primero* |

In [ ]:
print(f"{'Token':<25} {'Código':<8} {'Categoría'}")
print("-" * 55)
for token in documento:
    if not token.is_space and not token.is_punct:
        print(f"{token.text:<25} {token.pos_:<8} {spacy.explain(token.pos_)}")

---
### 2.2 Laboratorio de patrones

Los dos patrones del ejercicio anterior son solo un punto de partida. La celda siguiente acepta cualquier combinación de códigos de la tabla anterior.

Algunas combinaciones con interés lingüístico o terminológico:

| Patrón | Ejemplo esperado |
|--------|-----------------|
| `["NOUN", "ADJ"]` | *lingüística computacional*, *brecha tecnológica* |
| `["NOUN", "ADP", "NOUN"]` | *modelos de lengua*, *procesamiento de texto* |
| `["ADJ", "NOUN"]` | *alta calidad*, *gran corpus* |
| `["VERB", "NOUN"]` | *transforma la lingüística*, *requiere infraestructura* |
| `["ADV", "ADJ"]` | *radicalmente distinto*, *altamente especializado* |

Basta con modificar `mi_patron` para observar qué secuencias aparecen en el texto.

In [ ]:
mi_patron = ["NOUN", "ADJ"]  # Patrón a buscar: lista de códigos de la tabla

encontrados = []
longitud = len(mi_patron)

for posicion in range(len(documento) - longitud + 1):
    tokens_candidatos = [documento[posicion + j] for j in range(longitud)]
    if [t.pos_ for t in tokens_candidatos] == mi_patron:
        encontrados.append(" ".join(t.text for t in tokens_candidatos))

print(f"Patrón: {' + '.join(mi_patron)}")
print(f"Coincidencias: {len(encontrados)}\n")
for secuencia in encontrados:
    print(f"  - {secuencia}")

---
## 3. El fin de la forma: Similitud Semántica

### ¿Cómo ve la máquina el significado? (El Mapa de Palabras)

Para que una máquina "entienda" que *bulo* y *fake news* se parecen, no analiza sus letras. Lo que hace es convertir cada palabra en una **coordenada en un mapa gigante** (un espacio vectorial).

Considérese un mapa de solo dos ejes:
- **Eje X (Tecnología):** Cuanto más a la derecha, más tecnológico.
- **Eje Y (Veracidad):** Cuanto más arriba, más verdadero.

En este mapa mental:
1. **"Noticia"** estaría arriba (verdadero) y a la derecha (tecnológico si es digital).
2. **"Bulo"** estaría abajo (falso) y a la izquierda (general).
3. **"Fake news"** estaría abajo (falso) y a la derecha (tecnológico).

Al calcular la distancia entre los puntos, la máquina descubre que **"bulo"** y **"fake news"** comparten el mismo nivel de "falsedad" (eje Y), lo que las sitúa mucho más cerca entre sí que de la palabra **"televisor"** o **"noticia"**. 

**La magia no es tal:** es simplemente geometría. El modelo ha aprendido estas coordenadas leyendo millones de frases y observando qué palabras suelen aparecer en los mismos "barrios" del mapa.

**Nota:** Para este análisis se requiere un modelo con vectores cargados (`es_core_news_lg`), pero podemos simular la lógica con el modelo estándar para entender el concepto.

In [ ]:
# Comparación de similitud semántica
palabras = ["bulo", "noticia", "mentira", "periódico", "televisión"]

referencia = procesador("fake news")
comparaciones = []

for palabra in palabras:
    comparaciones.append({
        "Término": palabra,
        "Similitud con 'fake news'": referencia.similarity(procesador(palabra))
    })

tabla_similitud = pd.DataFrame(comparaciones).sort_values(by="Similitud con 'fake news'", ascending=False)
print(tabla_similitud)

### Reflexión profesional
- El sistema no busca letras comunes (como haríamos con Regex).
- El sistema busca **cercanía de uso**. 
- ¿Qué utilidad tiene esto para un profesional que busca el equivalente más natural en un contexto técnico?

---
## 4. Reto: El radar semántico

La tarea consiste en procesar un texto propio y encontrar qué términos orbitan alrededor de un concepto clave, sin que ese concepto sea mencionado explícitamente en todas las frases.

### Instrucciones:
1. Se define el `concepto_objetivo` en el código (por ejemplo, se cambia "tecnología" por "economía", "salud" o cualquier término que sea el eje del texto).
2. Se pega un texto extenso en `texto_usuario`.
3. Al ejecutar la celda, se obtiene qué palabras del texto tienen mayor afinidad semántica con el concepto.

In [ ]:
concepto_objetivo = "tecnología"
texto_usuario = """
PEGA AQUÍ TU TEXTO
"""

if texto_usuario.strip() == "PEGA AQUÍ TU TEXTO":
    print("El texto de muestra no ha sido reemplazado.")
else:
    documento_reto = procesador(texto_usuario)
    concepto_analizado = procesador(concepto_objetivo)

    analisis_semantico = []
    for token in documento_reto:
        if token.pos_ in ("NOUN", "ADJ", "VERB") and not token.is_stop and len(token.text) > 2:
            analisis_semantico.append({
                "Palabra": token.text,
                "Afinidad con el concepto": concepto_analizado.similarity(token)
            })

    tabla_reto = pd.DataFrame(analisis_semantico).sort_values(by="Afinidad con el concepto", ascending=False)
    
    print(f"--- Radar Semántico: '{concepto_objetivo}' ---\n")
    if not tabla_reto.empty:
        print(tabla_reto.drop_duplicates(subset=['Palabra']).head(15))
    else:
        print("No se encontraron palabras con carga semántica suficiente.")

---
### Nota sobre los resultados del radar semántico

Al ejecutar la celda anterior, el sistema emite un aviso:

```
UserWarning: [W007] The model you're using has no word vectors loaded,
so the result of the Doc.similarity method will be based on the tagger,
parser and NER, which may not give useful similarity judgements.
```

**Qué significa:** el modelo `es_core_news_sm` no incluye vectores de palabras. Cuando se llama a `similarity()`, el modelo no compara significados: compara representaciones internas basadas en el análisis gramatical (qué función cumple cada token en la frase). El número que devuelve existe, pero mide una cosa diferente a la cercanía semántica.

**Consecuencia práctica:** los resultados del radar pueden ser contraintuitivos. Dos palabras con significados opuestos podrían aparecer como más cercanas que dos sinónimos, simplemente porque el modelo las ha visto en posiciones sintácticas similares.

**Por qué se usa `sm` aquí:** para la extracción terminológica por patrones POS —que es el núcleo de esta sesión— el modelo pequeño es completamente suficiente. La limitación solo afecta a `similarity()`.

**La solución cuando los resultados importen:** sustituir `es_core_news_sm` por `es_core_news_md` en la celda de configuración. El modelo `md` incorpora vectores reales para ~20.000 palabras (44 MB frente a los 14 MB del `sm`) y produce similitudes semánticas fiables. El resto del código no cambia.

---
## Conclusiones

- La terminología profesional se extrae mediante **patrones gramaticales**, no solo frecuencias de palabras sueltas.
- Los **vectores** permiten que la máquina comprenda el parentesco entre palabras por su contexto de uso.
- Para el profesional de la lengua, estas herramientas permiten pasar de la búsqueda de texto a la **búsqueda de conceptos**.

### Nota técnica: Interpretación de avisos del sistema

Al ejecutar el código de similitud, el sistema devuelve un aviso (*UserWarning*) indicando que el modelo `es_core_news_sm` no incluye vectores de palabras pre-entrenados.

**Implicación técnica:**
- El modelo `sm` (small) es ligero y está optimizado para gramática y sintaxis. Para calcular la similitud, emplea tensores de contexto; esto ofrece una aproximación útil pero no la precisión semántica total de un mapa de palabras completo.
- Los modelos `md` (medium) o `lg` (large) incorporan cientos de miles de vectores reales. Esto permite una precisión semántica superior a cambio de un mayor consumo de memoria y tiempo de descarga (pasando de 15MB a más de 500MB).

**Criterio profesional:**
La elección del modelo debe responder al objetivo del proyecto. Para análisis morfosintáctico, el modelo reducido es suficiente. Para proyectos de terminología o semántica, se debe optar por modelos con vectores completos. Comprender estas limitaciones técnicas es fundamental para validar los resultados automáticos.

**Siguiente paso:** Estos vectores y patrones son los que permiten que modelos más grandes (LLMs) puedan generar texto con coherencia. Se retomará este hilo en el bloque de **IA Generativa**.